# CUDA integer ln approximation, range 1..65535

Compare the CUDA `tr_new_ln_scalar` path from `mrcp_quant/optimized_layers/common/tr_math.cuh` against a floating-point `log(x / 2**bits)` baseline over integer inputs `xq = 1..65535` with `BITS = 8`. The softmax extension exposes the CUDA kernel as `trptq_softmax_dp4a.ln_arith(...)`.

In [ ]:
from google.colab import drive
try:
    drive.mount('/content/drive')
except Exception as exc:
    print(f"Drive mount skipped: {exc}")

In [ ]:
import os
import sys
from pathlib import Path

import torch

PROJECT_DIR_PATH = globals().get("PROJECT_DIR_PATH", "/content/mrcp-tr-ptq")
PROJECT_DIR = Path(PROJECT_DIR_PATH).resolve()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

%cd {PROJECT_DIR_PATH}

BITS = 8
LN_SCALE = 1 << BITS
SOFTMAX_EXT_DIR = PROJECT_DIR / "mrcp_quant" / "optimized_layers" / "softmax"

In [ ]:
# Build/install after editing CUDA sources. Rerun this cell whenever softmax_cuda*.{cpp,cu}
# or common/tr_math.cuh changes, then restart/reload the notebook kernel if imports stay stale.
import shutil

softmax_dir = Path(SOFTMAX_EXT_DIR)
shutil.rmtree(softmax_dir / "build", ignore_errors=True)

for p in softmax_dir.glob("_trptq_softmax_dp4a*.so"):
    p.unlink()

!{sys.executable} -m pip install -v --no-build-isolation --no-cache-dir -e "{SOFTMAX_EXT_DIR}"

import _trptq_softmax_dp4a
print(_trptq_softmax_dp4a.__file__)

In [ ]:
import importlib

import trptq_softmax_dp4a
import _trptq_softmax_dp4a

trptq_softmax_dp4a = importlib.reload(trptq_softmax_dp4a)
print("Loaded:", trptq_softmax_dp4a.__file__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name())

if not hasattr(_trptq_softmax_dp4a, "ln_arith"):
    raise RuntimeError(
        "The loaded _trptq_softmax_dp4a binary is stale and does not export ln_arith. "
        "Run the install cell again, then restart this notebook kernel/runtime before continuing."
    )
print("ln_arith export: OK")

In [ ]:
def cuda_ln_arith(xq, bits=BITS):
    if xq.dtype != torch.int32:
        xq = xq.to(torch.int32)
    if not xq.is_cuda:
        xq = xq.cuda()
    out = torch.empty_like(xq, dtype=torch.int32)
    trptq_softmax_dp4a.ln_arith(xq.contiguous(), bits, out)
    return out

def cuda_ln_arith_opt(xq, out, bits=BITS):
    trptq_softmax_dp4a.ln_arith(xq.contiguous(), bits, out)
    return out


def safe_shift_new_ln_torch(x, shift):
    return torch.where(shift >= 0, x >> shift.clamp(min=0), x << (-shift).clamp(min=0))


def python_device_ln(xq, bits=BITS):
    xq = xq.to(torch.int32)
    aq = torch.floor(torch.log2(xq.float())).to(torch.int32) - bits
    k1 = safe_shift_new_ln_torch(xq, aq)
    k2 = (aq - 1) << bits
    k = k1 + k2
    return (k >> 1) + (k >> 3) + (k >> 4)


def baseline_ln_q(xq, bits=BITS):
    x = xq.float() / float(1 << bits)
    return torch.round(torch.log(x) * float(1 << bits)).to(torch.int32)


def summarize(name, approx_q, xq, baseline_q):
    approx_cpu = approx_q.detach().cpu().to(torch.int32)
    baseline_cpu = baseline_q.detach().cpu().to(torch.int32)
    x_cpu = xq.detach().cpu().to(torch.int32)
    diff_q = approx_cpu - baseline_cpu
    true_f = torch.log(x_cpu.float() / float(1 << BITS))
    approx_f = approx_cpu.float() / float(1 << BITS)
    diff_true = approx_f - true_f
    return {
        "name": name,
        "max_abs_q": diff_q.abs().max().item(),
        "mean_abs_q": diff_q.abs().float().mean().item(),
        "max_abs_float_vs_true": diff_true.abs().max().item(),
        "mean_abs_float_vs_true": diff_true.abs().mean().item(),
        "rmse_float_vs_true": diff_true.square().mean().sqrt().item(),
    }


def print_summary(rows):
    print(f"{'variant':<22} {'max_q':>8} {'mean_q':>10} {'max_true':>12} {'mean_true':>12} {'rmse_true':>12}")
    print("-" * 80)
    for row in rows:
        print(
            f"{row['name']:<22} {row['max_abs_q']:8.0f} {row['mean_abs_q']:10.4f} "
            f"{row['max_abs_float_vs_true']:12.4e} {row['mean_abs_float_vs_true']:12.4e} "
            f"{row['rmse_float_vs_true']:12.4e}"
        )

In [ ]:
assert torch.cuda.is_available(), "This notebook uses the CUDA extension; select a CUDA runtime first."
device = torch.device("cuda")

xq = torch.arange(1, 1 << 16, device=device, dtype=torch.int32)
baseline_q = baseline_ln_q(xq.cpu(), bits=BITS)
cuda_q = cuda_ln_arith(xq, bits=BITS)
python_q = python_device_ln(xq.cpu(), bits=BITS)

rows = [
    summarize("CUDA tr_new_ln", cuda_q, xq, baseline_q),
    summarize("Python mirror", python_q, xq.cpu(), baseline_q),
]
print_summary(rows)

mismatch = (cuda_q.cpu() != python_q).nonzero(as_tuple=False).flatten()
print(f"CUDA/Python mirror mismatches: {mismatch.numel()}")
if mismatch.numel():
    print("First mismatch indices:", mismatch[:10].tolist())

In [ ]:
xq_cpu = xq.cpu()
cuda_cpu = cuda_q.cpu()
baseline_cpu = baseline_q.cpu()
err = cuda_cpu - baseline_cpu

print(f"{'xq':>4} {'x':>9} {'base_q':>8} {'cuda_q':>8} {'err_q':>7} {'base_f':>10} {'cuda_f':>10}")
print("-" * 72)
sample_xq = list(range(1, 17))
for power in range(5, 16):
    center = 1 << power
    sample_xq.extend([center - 1, center, center + 1])
sample_xq.append((1 << 16) - 1)

indices = sorted({v - 1 for v in sample_xq if 1 <= v <= int(xq_cpu[-1])})

for i in indices:
    print(
        f"{int(xq_cpu[i]):4d} "
        f"{float(xq_cpu[i]) / LN_SCALE:9.5f} "
        f"{int(baseline_cpu[i]):8d} "
        f"{int(cuda_cpu[i]):8d} "
        f"{int(err[i]):7d} "
        f"{float(baseline_cpu[i]) / LN_SCALE:10.5f} "
        f"{float(cuda_cpu[i]) / LN_SCALE:10.5f}"
    )

In [ ]:
topk = torch.topk(err.abs(), k=15).indices

print(f"{'xq':>4} {'x':>9} {'base_q':>8} {'cuda_q':>8} {'err_q':>7} {'base_f':>10} {'cuda_f':>10}")
print("-" * 72)
for i in topk.tolist():
    print(
        f"{int(xq_cpu[i]):4d} "
        f"{float(xq_cpu[i]) / LN_SCALE:9.5f} "
        f"{int(baseline_cpu[i]):8d} "
        f"{int(cuda_cpu[i]):8d} "
        f"{int(err[i]):7d} "
        f"{float(baseline_cpu[i]) / LN_SCALE:10.5f} "
        f"{float(cuda_cpu[i]) / LN_SCALE:10.5f}"
    )

In [ ]:
# Compare CUDA tr_new_ln against torch.log on a large random tensor from the same range.
x_big = torch.randint(1, 1 << 16, (1_000_000,), device=device, dtype=torch.int32)
x_big_f = x_big.float() / float(LN_SCALE)


def benchmark_cuda(fn, warmup=20, repeats=200):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()

    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    for _ in range(repeats):
        fn()
    end.record()
    torch.cuda.synchronize()
    return start.elapsed_time(end) / repeats


tr_ln_ms = benchmark_cuda(lambda: cuda_ln_arith(x_big, bits=BITS))

x_big = x_big.cuda().contiguous()
out = torch.empty_like(x_big, dtype=torch.int32)
cuda_ln_arith_opt(x_big, out) 
torch_log_ms_opt = benchmark_cuda(lambda: torch.log(x_big_f))

torch_log_ms = benchmark_cuda(lambda: torch.log(x_big_f))

speedup = torch_log_ms / torch_log_ms_opt

print(f"{'kernel':<14} {'ms':>10}")
print("-" * 26)
print(f"{'tr_new_ln':<14} {tr_ln_ms:10.4f}")
print(f"{'tr_new_ln_opt':<14} {torch_log_ms_opt:10.4f}")
print(f"{'torch.log':<14} {torch_log_ms:10.4f}")
print(f"\nSpeedup: {speedup:.2f}x")